# Prismas e paralelepípedos

**Módulo 06 — Geometria Espacial** · Notebook 08 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0608a_prismas_e_paralelepipedos.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0602a_paralelismo.ipynb`](0602a_paralelismo.ipynb) e [`0603a_perpendicularidade.ipynb`](0603a_perpendicularidade.ipynb) (retas paralelas, planos paralelos e reta perpendicular a um plano), [`0604a_projecoes_distancias_e_angulos.ipynb`](0604a_projecoes_distancias_e_angulos.ipynb) (distância entre planos paralelos, ângulo entre reta e plano) e [`0607a_poliedros_convexos.ipynb`](0607a_poliedros_convexos.ipynb) (Relação de Euler e planificações)
- Módulo 04 — Geometria Plana: [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (Teorema de Pitágoras), [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (polígonos regulares e apótema) e [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (áreas de polígonos)
- Módulo 05 — Trigonometria: [`0509a_triangulos_retangulos.ipynb`](../05_trigonometria/0509a_triangulos_retangulos.ipynb) (razões trigonométricas)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- identificar os elementos de um prisma e classificá-lo (reto, oblíquo, regular), reconhecendo paralelepípedos, paralelepípedos retângulos, cubos e romboedros;
- calcular diagonais e áreas do cubo e do paralelepípedo retângulo;
- calcular as áreas lateral e total de um prisma, usando a planificação;
- descrever os polígonos que aparecem quando se corta um cubo por um plano.

**Tempo estimado:** 80 a 100 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math
import numpy as np

## A vassoura cabe na caixa?

Dia de mudança. Você tem uma caixa de papelão de 1,0 m de comprimento, 0,8 m de largura e 0,6 m de altura, e precisa guardar um cabo de vassoura de 1,3 m. Nenhuma das medidas da caixa chega a 1,3 m. Então não cabe... certo?

Não necessariamente! Dentro de uma caixa, o maior segmento possível não fica deitado no fundo nem encostado numa parede: ele vai de um canto de baixo até o canto **oposto** de cima. É a **diagonal** da caixa.

🕹️ Mexa nas medidas da caixa e no tamanho da vassoura.

In [2]:
def guardar_vassoura(
    comprimento: float, largura: float, altura: float, vassoura: float
) -> None:
    """Desenha a caixa, a diagonal AG e a vassoura apoiada no canto A."""
    v = u3d.vertices_do_paralelepipedo(comprimento, largura, altura)
    caixa = u3d.paralelepipedo_retangulo(comprimento, largura, altura)

    # A diagonal da caixa (de onde vem esta fórmula? Você vai descobrir na seção 3).
    diagonal = math.sqrt(comprimento**2 + largura**2 + altura**2)
    cabe = vassoura <= diagonal

    # A vassoura sai do canto A (que fica na origem) na direção de G. Para ela ter o
    # comprimento escolhido, andamos a fração vassoura / diagonal do caminho de A até
    # G (as coordenadas servem só para desenhar).
    fracao = vassoura / diagonal
    ponta = [fracao * coordenada for coordenada in v["G"]]

    fig = u3d.criar_figura(titulo="A vassoura cabe na caixa?")
    fig = u3d.desenhar_poliedro(fig, *caixa, opacidade=0.2, rotulos=list(v))
    fig = u3d.desenhar_segmento(
        fig, v["A"], v["G"], cor=u3d.CORES["apoio"], tracejado=True
    )
    cor = u3d.CORES["reta"] if cabe else u3d.CORES["destaque"]
    fig = u3d.desenhar_segmento(fig, v["A"], ponta, cor=cor, espessura=10)
    u3d.mostrar(fig)

    print(f"Diagonal da caixa: {diagonal:.2f} m")
    if cabe:
        print("✔ A vassoura cabe, inclinada ao longo da diagonal!")
    else:
        print("✘ A vassoura não cabe nem na diagonal.")


def medida(descricao: str, minimo: float, maximo: float, valor: float) -> widgets.FloatSlider:
    """Cria um slider de medida, em metros, com passo de 0,1 m."""
    return widgets.FloatSlider(
        min=minimo, max=maximo, step=0.1, value=valor, description=descricao,
        continuous_update=False,
    )


interact(
    guardar_vassoura,
    comprimento=medida("comprimento", 0.4, 1.5, 1.0),
    largura=medida("largura", 0.4, 1.5, 0.8),
    altura=medida("altura", 0.2, 1.5, 0.6),
    vassoura=medida("vassoura", 0.5, 2.0, 1.3),
);

interactive(children=(FloatSlider(value=1.0, continuous_update=False, description='comprimento', max=1.5, min=…

A caixa é um **paralelepípedo retângulo**, um caso especial de **prisma**. Neste notebook você vai descobrir de onde vem a fórmula da diagonal, vai calcular quanto papelão é necessário para fazer uma caixa (a **área**) e vai ver o que aparece quando se corta um cubo com uma faca bem afiada.

## 1. Prismas

### Prisma ilimitado

Pegue um polígono convexo num plano $\alpha$ e uma reta $r$ que fura $\alpha$. Por **cada** ponto do polígono, trace a reta paralela a $r$. O sólido formado por todas essas retas é um **prisma ilimitado**: uma espécie de "tubo" infinito com seção em forma de polígono.

- As retas que passam pelos vértices do polígono são as **arestas** do prisma ilimitado.
- Um plano que corta todas as arestas determina uma **seção**; se ele for **perpendicular** às arestas, é uma **seção reta**.
- Seções feitas por planos **paralelos** são polígonos **congruentes** (seus lados correspondentes são lados opostos de paralelogramos).

### Prisma

> 📌 **Definição.** Um **prisma** é a parte de um prisma ilimitado compreendida entre dois planos paralelos que cortam todas as arestas.

**Elementos**
- **Bases:** os dois polígonos congruentes, em planos paralelos.
- **Faces laterais:** paralelogramos.
- **Arestas laterais:** paralelas e de mesmo comprimento.
- **Arestas das bases:** os lados das bases.
- **Altura** $h$: a distância entre os planos das bases (notebook 04).

O nome do prisma vem do polígono da base: **triangular**, **quadrangular**, **pentagonal**, **hexagonal**...

> 📎 **Conexão.** Um prisma de base com $n$ lados tem $V = 2n$, $A = 3n$ e $F = n + 2$. Confira a Relação de Euler (notebook 07): $2n - 3n + (n + 2) = 2$. ✔

### Classificação

| Tipo | Arestas laterais | Faces laterais | Altura |
|---|---|---|---|
| **reto** | perpendiculares às bases | retângulos | igual à aresta lateral |
| **oblíquo** | oblíquas às bases | paralelogramos | menor que a aresta lateral |
| **regular** | **reto** e com base em polígono regular | retângulos congruentes | igual à aresta lateral |

No prisma oblíquo, se a aresta lateral $\ell$ forma um ângulo $\theta$ com o plano da base, a altura é $h = \ell \cdot \sin\theta$ (notebook 04).

> ⚠️ **Erro comum.** Chamar de "regular" qualquer prisma de base regular. Um prisma oblíquo com base hexagonal regular **não** é regular: para ser regular, o prisma precisa ser **reto**.

🎯 **No dia a dia:** a embalagem de chocolate triangular (prisma triangular), caixas e tijolos (prismas quadrangulares), lápis sextavados e porcas de parafuso (prismas hexagonais), e os prismas de vidro que separam a luz branca nas cores do arco-íris.

🕹️ Monte prismas no widget abaixo. A base é um polígono regular de lado 2; a `inclinacao` é o ângulo $\theta$ entre a aresta lateral e o plano da base.

In [3]:
def contar_elementos(vertices: list, faces: list) -> tuple[int, int, int]:
    """Conta vértices (V), arestas (A) e faces (F) de um poliedro (como no notebook 07)."""
    arestas = set()
    for face in faces:
        for k in range(len(face)):
            i = face[k]
            j = face[(k + 1) % len(face)]  # o próximo vértice (o último se liga ao primeiro)
            arestas.add(tuple(sorted((i, j))))  # (2, 5) e (5, 2) são a mesma aresta
    return len(vertices), len(arestas), len(faces)


NOMES_DOS_PRISMAS = {
    3: "triangular", 4: "quadrangular", 5: "pentagonal", 6: "hexagonal",
    7: "heptagonal", 8: "octogonal",
}


def montar_prisma(
    lados_da_base: int, aresta_lateral: float, inclinacao: float,
    mostrar_secao_reta: bool,
) -> None:
    """Desenha um prisma de base regular, com a altura e, se pedido, a seção reta."""
    base = u3d.poligono_regular(lados_da_base, lado=2)
    # A aresta lateral tem comprimento ℓ e forma o ângulo θ com o plano da base: ela
    # "anda" ℓ · cos θ na horizontal e ℓ · sen θ na vertical (razões trigonométricas).
    theta = math.radians(inclinacao)
    deslocamento = (aresta_lateral * math.cos(theta), 0, aresta_lateral * math.sin(theta))
    vertices, faces = u3d.prisma(base, deslocamento)

    # A altura é a distância entre os planos das bases (notebook 04).
    altura = aresta_lateral * math.sin(math.radians(inclinacao))

    nome = NOMES_DOS_PRISMAS[lados_da_base]
    fig = u3d.criar_figura(titulo=f"Prisma {nome}")
    fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.3)

    # A altura: do centro da base de baixo, subindo na vertical até o plano da base
    # de cima, onde forma um ângulo reto.
    pe, topo = (0, 0, 0), (0, 0, altura)
    fig = u3d.desenhar_segmento(
        fig, pe, topo, rotulo="h", cor=u3d.CORES["reta"], tracejado=True
    )
    fig = u3d.desenhar_angulo_reto(fig, topo, pe, (1, 0, altura), cor=u3d.CORES["reta"])
    if inclinacao < 90:
        # No prisma oblíquo, o topo da altura fica fora da base de cima: ligamos o
        # topo ao centro da base de cima, por dentro do plano dessa base.
        fig = u3d.desenhar_segmento(
            fig, topo, (deslocamento[0], 0, altura), cor=u3d.CORES["apoio"],
            espessura=2, tracejado=True,
        )
        # O ângulo θ entre a aresta lateral do vértice 0 e a sua sombra na base.
        sombra = (vertices[0][0] + deslocamento[0], 0, 0)
        fig = u3d.desenhar_segmento(
            fig, vertices[0], sombra, cor=u3d.CORES["apoio"], espessura=2,
            tracejado=True,
        )
        fig = u3d.desenhar_angulo(
            fig, vertices[0], sombra, vertices[lados_da_base], rotulo="θ",
            cor=u3d.CORES["reta_secundaria"],
        )

    if mostrar_secao_reta:
        # A seção reta é perpendicular às arestas laterais. Para encontrá-la, cortamos
        # um prisma auxiliar bem comprido, com as mesmas arestas prolongadas: um
        # pedaço do "prisma ilimitado". O plano passa pelo centro do prisma.
        recuo = [-2 * c for c in deslocamento]
        base_auxiliar = u3d.poligono_regular(lados_da_base, lado=2, centro=recuo)
        auxiliar = u3d.prisma(base_auxiliar, [5 * c for c in deslocamento])
        centro = [c / 2 for c in deslocamento]
        secao_reta = u3d.secao_plana(*auxiliar, (centro, deslocamento))
        fig = u3d.desenhar_poligono(
            fig, secao_reta, cor=u3d.CORES["destaque"], opacidade=0.6
        )
        # A seção cabe no prisma se todos os seus pontos estão entre as duas bases.
        cabe = all(-1e-9 <= ponto[2] <= altura + 1e-9 for ponto in secao_reta)
        if not cabe:
            # Mostramos as arestas laterais prolongadas, onde a seção se apoia.
            for k in range(lados_da_base):
                meio = [(a + b) / 2 for a, b in zip(vertices[k], vertices[k + lados_da_base])]
                fig = u3d.desenhar_reta(
                    fig, meio, deslocamento, cor=u3d.CORES["apoio"],
                    comprimento=5 * aresta_lateral, espessura=1, tracejado=True,
                )
    u3d.mostrar(fig)

    V, A, F = contar_elementos(vertices, faces)
    print(f"V = {V}, A = {A}, F = {F}   (V − A + F = {V - A + F})")
    print(f"Aresta lateral: ℓ = {aresta_lateral:.2f}")
    print(f"Altura: h = ℓ · sen θ = {aresta_lateral:.2f} · sen {inclinacao:g}° = {altura:.2f}")
    if inclinacao == 90:
        print(f"Classificação: prisma {nome} reto e regular")
    else:
        print(f"Classificação: prisma {nome} oblíquo")
    if mostrar_secao_reta and not cabe:
        print(
            "Neste prisma baixo e inclinado, a seção reta pelo centro sai pelas bases: "
            "ela só cabe no prisma ilimitado (as arestas prolongadas, tracejadas)."
        )


interact(
    montar_prisma,
    lados_da_base=widgets.IntSlider(
        min=3, max=8, step=1, value=6, description="lados da base",
        continuous_update=False,
    ),
    aresta_lateral=widgets.FloatSlider(
        min=1, max=5, step=0.5, value=4, description="aresta lateral",
        continuous_update=False,
    ),
    inclinacao=widgets.FloatSlider(
        min=30, max=90, step=10, value=90, description="inclinação (°)",
        continuous_update=False,
    ),
    mostrar_secao_reta=widgets.Checkbox(value=False, description="Mostrar seção reta"),
);

interactive(children=(IntSlider(value=6, continuous_update=False, description='lados da base', max=8, min=3), …

**Pare e pense**

1. Quantos vértices, arestas e faces tem um prisma pentagonal?
2. Num prisma oblíquo, a aresta lateral mede 10 cm e forma 30° com a base. Qual é a altura?

<details>
<summary>Ver respostas</summary>

1. $V = 10$, $A = 15$, $F = 7$.
2. $h = 10 \cdot \sin 30° = 5$ cm.
</details>

## 2. Paralelepípedos e romboedros

> 📌 **Definição.**
> - Um **paralelepípedo** é um prisma cujas bases são paralelogramos. Todas as suas 6 faces são paralelogramos.
> - Um **paralelepípedo reto** tem as arestas laterais perpendiculares às bases.
> - Um **paralelepípedo retângulo** (ou **ortoedro**) tem as 6 faces retangulares. É o formato de uma caixa de sapatos.
> - Um **cubo** é um paralelepípedo retângulo com as 12 arestas iguais.
> - Um **romboedro** é um paralelepípedo com as 6 faces losangos congruentes.

Repare na "família":

- paralelepípedo
  - paralelepípedo reto
    - paralelepípedo retângulo
      - **cubo**
  - romboedro
    - **cubo**

O cubo é, ao mesmo tempo, um paralelepípedo retângulo e um romboedro (um losango com ângulos retos é um quadrado).

**Propriedades de todo paralelepípedo**
1. Faces opostas são **paralelas e congruentes**. Por isso, **qualquer** par de faces opostas pode ser tomado como bases.
2. As quatro **diagonais** ($AG$, $BH$, $CE$, $DF$) se cortam num mesmo ponto, que é o **ponto médio** de cada uma delas: o **centro** do paralelepípedo.
3. No paralelepípedo **retângulo**, as quatro diagonais têm o **mesmo comprimento**.

🎯 **No dia a dia:** caixas, tijolos, contêineres, livros, aquários. Os romboedros aparecem na natureza: a **calcita**, um mineral muito comum, se quebra naturalmente em pedaços com a forma de romboedros.

🕹️ Explore a família no widget. O `angulo` vale para o paralelepípedo oblíquo (inclinação da aresta lateral em relação à base) e para o romboedro (ângulo entre as arestas que saem de $A$).

In [ ]:
DIAGONAIS = ["AG", "BH", "CE", "DF"]
CORES_DAS_DIAGONAIS = [
    u3d.CORES["reta"], u3d.CORES["reta_secundaria"], u3d.CORES["plano_secundario"],
    u3d.CORES["plano_terciario"],
]


def explorar_familia(tipo: str, angulo: float, mostrar_diagonais: bool) -> None:
    """Desenha um membro da família dos paralelepípedos e mede as suas diagonais."""
    if tipo == "Paralelepípedo oblíquo":
        # A aresta lateral AE (comprimento 2,5) forma o ângulo escolhido com a base.
        theta = math.radians(angulo)
        aresta_lateral = (2.5 * math.cos(theta), 0, 2.5 * math.sin(theta))
        vertices, faces = u3d.paralelepipedo((0, 0, 0), (3, 0, 0), (1, 2, 0), aresta_lateral)
    elif tipo == "Paralelepípedo retângulo":
        vertices, faces = u3d.paralelepipedo_retangulo(3, 2, 1.5)
    elif tipo == "Cubo":
        vertices, faces = u3d.paralelepipedo_retangulo(2, 2, 2)
    else:
        vertices, faces = u3d.romboedro(2, angulo)
    nomes = dict(zip("ABCDEFGH", vertices))

    fig = u3d.criar_figura(titulo=tipo)
    fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.3, rotulos=list(nomes))
    # O centro é a média dos 8 vértices (as coordenadas servem só para desenhar).
    centro = np.mean(vertices, axis=0)
    if mostrar_diagonais:
        for diagonal, cor in zip(DIAGONAIS, CORES_DAS_DIAGONAIS):
            fig = u3d.desenhar_segmento(
                fig, nomes[diagonal[0]], nomes[diagonal[1]], cor=cor, espessura=5
            )
        fig = u3d.desenhar_ponto(fig, centro, rotulo="O")
    u3d.mostrar(fig)

    print("Diagonal   comprimento   O até uma ponta   O até a outra")
    for diagonal in DIAGONAIS:
        inicio, fim = nomes[diagonal[0]], nomes[diagonal[1]]
        comprimento = u3d.medir_distancia(inicio, fim)
        ate_inicio = u3d.medir_distancia(centro, inicio)
        ate_fim = u3d.medir_distancia(centro, fim)
        print(f"{diagonal:>8}{comprimento:>14.2f}{ate_inicio:>18.2f}{ate_fim:>16.2f}")


interact(
    explorar_familia,
    tipo=widgets.Dropdown(
        options=[
            "Paralelepípedo oblíquo", "Paralelepípedo retângulo", "Cubo", "Romboedro",
        ],
        description="tipo",
    ),
    angulo=widgets.FloatSlider(
        min=40, max=110, step=10, value=70, description="ângulo (°)",
        continuous_update=False,
    ),
    mostrar_diagonais=widgets.Checkbox(value=True, description="Mostrar diagonais"),
);

interactive(children=(Dropdown(description='tipo', options=('Paralelepípedo oblíquo', 'Paralelepípedo retângul…

**Pare e pense**

1. Todo cubo é um romboedro? Todo romboedro é um cubo?
2. No widget, em quais tipos as quatro diagonais têm o mesmo comprimento?

<details>
<summary>Ver respostas</summary>

1. Todo cubo é um romboedro (losangos com ângulos retos são quadrados). Mas nem todo romboedro é cubo: as faces podem ser losangos sem ângulos retos.
2. No paralelepípedo retângulo e no cubo. No oblíquo e no romboedro (não cúbico), elas têm comprimentos diferentes, mas continuam se cortando ao meio.
</details>

## 3. Diagonal e área do paralelepípedo retângulo e do cubo

### A diagonal

Considere um paralelepípedo retângulo de dimensões $a$, $b$ e $c$ (com $AB = a$, $AD = b$, $AE = c$). Queremos a diagonal $d = BH$.

1. **Diagonal da base.** No retângulo $ABCD$, a diagonal $BD$ é a hipotenusa de um triângulo retângulo de catetos $a$ e $b$:
   $$BD^2 = a^2 + b^2$$
2. **Diagonal do paralelepípedo.** A aresta $DH$ é perpendicular ao plano da base (notebook 03). Então $DH$ forma ângulo reto com $BD$, e o triângulo $BDH$ é retângulo em $D$:
   $$d^2 = BD^2 + DH^2 = a^2 + b^2 + c^2$$

$$\boxed{d = \sqrt{a^2 + b^2 + c^2}}$$

### A área total

As 6 faces formam três pares de retângulos iguais:
$$\boxed{S = 2(ab + ac + bc)}$$

### O cubo

No cubo de aresta $a$, basta fazer $a = b = c$:
- diagonal da face: $a\sqrt{2}$;
- diagonal do cubo: $d = \sqrt{3a^2} = a\sqrt{3}$;
- área total: $S = 6a^2$.

> 💡 **Dica.** Existe uma relação bonita entre a soma das três dimensões, a diagonal e a área:
> $$(a + b + c)^2 = a^2 + b^2 + c^2 + 2(ab + ac + bc) = d^2 + S$$
> Ela permite calcular a área total sem conhecer cada dimensão separadamente.

🕹️ Mexa nas dimensões e acompanhe os dois triângulos retângulos.

In [ ]:
def calcular_diagonal(a: float, b: float, c: float) -> None:
    """Desenha os dois triângulos retângulos da dedução e calcula d e S."""
    v = u3d.vertices_do_paralelepipedo(a, b, c)
    B, D, H = v["B"], v["D"], v["H"]

    fig = u3d.criar_figura(titulo="Dois triângulos retângulos: BAD e BDH")
    fig = u3d.desenhar_poliedro(
        fig, *u3d.paralelepipedo_retangulo(a, b, c), opacidade=0.15, rotulos=list(v)
    )
    # 1º triângulo: BAD, na base, retângulo em A; a hipotenusa é a diagonal BD.
    fig = u3d.desenhar_poligono(fig, [B, v["A"], D], cor=u3d.CORES["reta"], opacidade=0.3)
    fig = u3d.desenhar_angulo_reto(fig, v["A"], B, D, cor=u3d.CORES["reta"])
    fig = u3d.desenhar_segmento(fig, B, D, cor=u3d.CORES["reta"], espessura=6)
    # 2º triângulo: BDH, "em pé", retângulo em D; a hipotenusa é a diagonal BH.
    fig = u3d.desenhar_poligono(
        fig, [B, D, H], cor=u3d.CORES["plano_terciario"], opacidade=0.4
    )
    fig = u3d.desenhar_angulo_reto(fig, D, B, H, cor=u3d.CORES["plano_terciario"])
    fig = u3d.desenhar_segmento(fig, B, H, cor=u3d.CORES["destaque"], espessura=6)
    u3d.mostrar(fig)

    diagonal_base = math.sqrt(a**2 + b**2)
    diagonal = math.sqrt(diagonal_base**2 + c**2)  # = √(a² + b² + c²)
    area_total = 2 * (a * b + a * c + b * c)

    print(f"Diagonal da base: BD = √(a² + b²) = {diagonal_base:.2f}")
    print(f"Diagonal: d = BH = √(BD² + c²) = {diagonal:.2f}")
    print(f"Área total: S = 2(ab + ac + bc) = {area_total:.2f}")
    print(
        f"Conferência com o utils_3d: BH = {u3d.medir_distancia(B, H):.2f} e "
        f"S = {u3d.medir_area_total(*u3d.paralelepipedo_retangulo(a, b, c)):.2f}"
    )
    print(f"(a + b + c)² = {(a + b + c)**2:.2f} e d² + S = {diagonal**2 + area_total:.2f}")


def dimensao(nome: str, valor: float) -> widgets.FloatSlider:
    """Cria um slider de 1 a 6 para uma das dimensões."""
    return widgets.FloatSlider(
        min=1, max=6, step=0.5, value=valor, description=nome, continuous_update=False
    )


interact(calcular_diagonal, a=dimensao("a", 2), b=dimensao("b", 3), c=dimensao("c", 6));

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='a', max=6.0, min=1.0, step=…

**Pare e pense**

1. Um cubo tem aresta 3. Quanto medem a diagonal de uma face e a diagonal do cubo?
2. Um paralelepípedo retângulo tem dimensões 3, 4 e 12. Quanto mede a diagonal?

<details>
<summary>Ver respostas</summary>

1. $3\sqrt{2} \approx 4{,}24$ e $3\sqrt{3} \approx 5{,}20$.
2. $\sqrt{9 + 16 + 144} = \sqrt{169} = 13$.
</details>

## 4. Área lateral e área total do prisma

> 📌 **Definição.**
> - A **área lateral** $S_\ell$ de um prisma é a soma das áreas das faces laterais.
> - A **área total** é a área lateral mais as áreas das duas bases: $S_t = S_\ell + 2 S_b$.

### Prisma reto: a faixa de retângulos

Planifique a superfície lateral de um prisma reto: as faces laterais viram retângulos lado a lado, todos com a mesma altura $h$, formando um **retângulo grande** cuja base é o **perímetro** da base do prisma ($2p$):
$$S_\ell = 2p \cdot h$$

Para as bases, use as áreas da Geometria Plana. Para polígonos regulares de lado $\ell$:

| Base | Área |
|---|---|
| triângulo equilátero | $\dfrac{\ell^2 \sqrt{3}}{4}$ |
| quadrado | $\ell^2$ |
| hexágono regular | $6 \cdot \dfrac{\ell^2 \sqrt{3}}{4} = \dfrac{3\ell^2\sqrt{3}}{2}$ |

### Prisma oblíquo: a seção reta

Num prisma oblíquo, as faces laterais são paralelogramos. Cada um tem como "base" uma aresta lateral (comprimento $\ell$) e como altura o lado correspondente da **seção reta** (que é perpendicular às arestas laterais). Somando:
$$S_\ell = 2p_{\text{seção reta}} \cdot \ell$$
em que $2p_{\text{seção reta}}$ é o perímetro da seção reta. No prisma reto, a seção reta é a própria base e $\ell = h$: a fórmula vira a anterior.

> 🕵️ **Curiosidade histórica.** As abelhas constroem os favos de mel com células que são prismas hexagonais. O matemático grego **Papo de Alexandria** (século IV) já observava que o hexágono é o formato que permite dividir uma superfície em partes iguais gastando menos "parede". Essa ideia só foi demonstrada rigorosamente em **1999**, pelo matemático americano **Thomas Hales**: é a chamada "conjectura do favo de mel".

🎯 **Aplicações:** a quantidade de papelão de uma embalagem (área total), o número de azulejos para revestir uma piscina (área lateral mais o fundo), a quantidade de tinta para pintar uma coluna (área lateral).

🕹️ Dobre e desdobre o prisma regular abaixo e compare as áreas.

In [6]:
def planificar_prisma(lados_da_base: int, lado: float, altura: float, dobra: float) -> None:
    """Dobra a planificação de um prisma regular e calcula as suas áreas."""
    planificacao = u3d.planificacao_de_prisma_regular(lados_da_base, lado, altura)
    faces = u3d.dobrar_planificacao(planificacao, dobra)
    # A faixa lateral numa cor e as duas bases em outra.
    cores = [u3d.CORES["plano"]] * lados_da_base + [u3d.CORES["plano_secundario"]] * 2
    fig = u3d.criar_figura(titulo="Planificação do prisma regular")
    fig = u3d.desenhar_faces(fig, faces, cores=cores)
    if dobra == 0:
        fig = u3d.definir_camera(fig, olho=(0, 0, 2.5))  # vista de cima
    u3d.mostrar(fig)

    perimetro = lados_da_base * lado
    area_lateral = perimetro * altura
    apotema = lado / (2 * math.tan(math.pi / lados_da_base))  # apótema do polígono regular (módulo 04)
    area_base = perimetro * apotema / 2  # área do polígono regular = semiperímetro × apótema
    area_total = area_lateral + 2 * area_base

    print(f"Perímetro da base: 2p = {lados_da_base} × {lado:g} = {perimetro:.2f}")
    print(f"Área lateral: S_ℓ = 2p · h = {perimetro:.2f} × {altura:g} = {area_lateral:.2f}")
    print(f"Apótema da base: {apotema:.2f}")
    print(f"Área de uma base: S_b = p · apótema = {area_base:.2f}")
    print(f"Área total: S_t = S_ℓ + 2 S_b = {area_total:.2f}")
    prisma = u3d.prisma(u3d.poligono_regular(lados_da_base, lado=lado), (0, 0, altura))
    print(f"Conferência com o utils_3d: área total = {u3d.medir_area_total(*prisma):.2f}")


interact(
    planificar_prisma,
    lados_da_base=widgets.IntSlider(
        min=3, max=8, step=1, value=5, description="lados da base",
        continuous_update=False,
    ),
    lado=widgets.FloatSlider(
        min=1, max=3, step=0.5, value=2, description="lado", continuous_update=False
    ),
    altura=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=4, description="altura", continuous_update=False
    ),
    dobra=widgets.FloatSlider(
        min=0, max=1, step=0.1, value=0, description="Dobrar", continuous_update=False
    ),
);

interactive(children=(IntSlider(value=5, continuous_update=False, description='lados da base', max=8, min=3), …

### Conferindo a fórmula do prisma oblíquo

O prisma abaixo é oblíquo, com base hexagonal regular de lado 2. A seção reta (em destaque) é perpendicular às arestas laterais e passa pelo centro do prisma. Vamos calcular a área lateral de dois jeitos: pela fórmula $S_\ell = 2p_{\text{seção reta}} \cdot \ell$ e medindo todas as faces com o `utils_3d`.

In [7]:
base = u3d.poligono_regular(6, lado=2)
deslocamento = (2, 0, 4)  # o vetor de cada aresta lateral: o prisma "tomba" para a frente
vertices, faces = u3d.prisma(base, deslocamento)

# O plano da seção reta é perpendicular às arestas laterais e passa pelo centro do
# prisma, na metade do caminho entre os centros das bases (só para desenhar).
centro = (1, 0, 2)
secao_reta = u3d.secao_plana(vertices, faces, (centro, deslocamento))

# Perímetro da seção reta: a soma dos seus lados (o último vértice volta ao primeiro).
perimetro_secao = 0
for k in range(len(secao_reta)):
    perimetro_secao += u3d.medir_distancia(
        secao_reta[k], secao_reta[(k + 1) % len(secao_reta)]
    )
aresta = u3d.medir_distancia(vertices[0], vertices[6])  # o vértice 6 fica acima do 0
area_lateral_formula = perimetro_secao * aresta

# Conferência: área total medida, menos as duas bases.
area_da_base = u3d.medir_area_do_poligono(base)
area_lateral_medida = u3d.medir_area_total(vertices, faces) - 2 * area_da_base

fig = u3d.criar_figura(titulo="Prisma oblíquo e a sua seção reta")
fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.3)
fig = u3d.desenhar_poligono(fig, secao_reta, cor=u3d.CORES["destaque"], opacidade=0.7)
u3d.mostrar(fig)

print(f"Perímetro da seção reta: 2p = {perimetro_secao:.4f}")
print(f"Aresta lateral: ℓ = {aresta:.4f}")
print(f"Área lateral pela fórmula (2p · ℓ):  {area_lateral_formula:.4f}")
print(f"Área lateral medida face a face:     {area_lateral_medida:.4f}")

Perímetro da seção reta: 2p = 11.3751
Aresta lateral: ℓ = 4.4721
Área lateral pela fórmula (2p · ℓ):  50.8712
Área lateral medida face a face:     50.8712


**Pare e pense**

1. Um prisma hexagonal regular tem aresta da base 2 e altura 5. Qual é a área lateral?
2. Por que, no prisma oblíquo, usamos o perímetro da seção reta e não o perímetro da base?

<details>
<summary>Ver respostas</summary>

1. $S_\ell = 6 \cdot 2 \cdot 5 = 60$.
2. Porque as faces laterais são paralelogramos: a área de cada um é "base × altura", e a altura de cada paralelogramo (medida perpendicularmente à aresta lateral) é um lado da seção reta, não um lado da base.
</details>

## 5. Seções planas do cubo

O que aparece quando você corta um cubo de queijo com uma faca, num corte reto (plano)? Depende de como você corta!

- Um corte paralelo a uma face dá um **quadrado**.
- Um corte que passa por duas arestas opostas dá um **retângulo**.
- Um corte perto de um vértice, "arrancando o canto", dá um **triângulo**.
- Cortes mais "inclinados" podem dar **pentágonos** e até **hexágonos**.

**Quantos lados, no máximo?** Cada lado da seção está numa face do cubo, e o plano encontra cada face no máximo uma vez (num segmento). Como o cubo tem 6 faces, a seção tem **no máximo 6 lados**. Nunca aparece um heptágono!

**O hexágono regular escondido.** Se você cortar o cubo por um plano **perpendicular a uma diagonal**, passando pelo **centro**, o plano passa pelos pontos médios de seis arestas e a seção é um **hexágono regular**. Movendo esse plano ao longo da diagonal, a seção começa como um ponto, vira um triângulo equilátero, depois um hexágono, e volta a ser triângulo.

> 📎 **Conexão.** No notebook 03 você viu que a diagonal $AG$ é perpendicular ao plano $BDE$. O triângulo equilátero $BDE$ é uma das seções dessa família.

🕹️ Explore os dois modos do widget: o corte perpendicular à diagonal (use o controle `posicao`) e o corte livre (use os controles de inclinação e o `deslocamento`). O cubo tem aresta 2.

In [7]:
CUBO = u3d.paralelepipedo_retangulo(2, 2, 2)
VERTICES_DO_CUBO = u3d.vertices_do_paralelepipedo(2, 2, 2)
NOMES_DAS_SECOES = {3: "triângulo", 4: "quadrilátero", 5: "pentágono", 6: "hexágono"}


def cortar_cubo(
    modo: str, posicao: float, inclinacao_1: float, inclinacao_2: float,
    deslocamento: float,
) -> None:
    """Corta o cubo por um plano e descreve o polígono da seção."""
    if modo == "Perpendicular à diagonal AG":
        # O plano passa pelo ponto da diagonal AG escolhido pela posição (de 0, em A,
        # até 6, em G) e é perpendicular a AG.
        ponto = (posicao / 3, posicao / 3, posicao / 3)
        normal = (1, 1, 1)
    else:
        # Dois ângulos orientam a "direção" do plano, como latitude e longitude num
        # globo; seno e cosseno transformam esses ângulos numa direção no espaço.
        a1, a2 = math.radians(inclinacao_1), math.radians(inclinacao_2)
        normal = (math.cos(a1) * math.sin(a2), math.sin(a1), math.cos(a1) * math.cos(a2))
        # O plano fica à distância `deslocamento` do centro (1, 1, 1) do cubo.
        ponto = tuple(1 + deslocamento * n for n in normal)
    secao = u3d.secao_plana(*CUBO, (ponto, normal))

    fig = u3d.criar_figura(titulo=modo)
    fig = u3d.desenhar_poliedro(fig, *CUBO, opacidade=0.15, rotulos=list(VERTICES_DO_CUBO))
    if modo == "Perpendicular à diagonal AG":
        fig = u3d.desenhar_segmento(
            fig, VERTICES_DO_CUBO["A"], VERTICES_DO_CUBO["G"], cor=u3d.CORES["apoio"],
            tracejado=True,
        )
    fig = u3d.desenhar_plano(fig, ponto, normal, opacidade=0.1, tamanho=4.5)
    if secao:
        fig = u3d.desenhar_poligono(fig, secao, cor=u3d.CORES["destaque"], opacidade=0.6)
    u3d.mostrar(fig)

    if not secao:
        print("O plano não corta o cubo (ou só toca num vértice ou numa aresta).")
        return
    lados = [
        u3d.medir_distancia(secao[k], secao[(k + 1) % len(secao)])
        for k in range(len(secao))
    ]
    print(f"Seção: {NOMES_DAS_SECOES[len(secao)]} ({len(secao)} lados)")
    print("Lados:", ", ".join(f"{lado:.2f}" for lado in lados))
    print(f"Área: {u3d.medir_area_do_poligono(secao):.2f}")
    if max(lados) - min(lados) < 1e-6:
        print("Todos os lados iguais!")


def angulo_do_corte(descricao: str) -> widgets.FloatSlider:
    """Cria um slider de −80° a 80° para orientar o plano do corte livre."""
    return widgets.FloatSlider(
        min=-80, max=80, step=10, value=20, description=descricao,
        continuous_update=False,
    )


interact(
    cortar_cubo,
    modo=widgets.RadioButtons(
        options=["Perpendicular à diagonal AG", "Corte livre"], description="modo"
    ),
    posicao=widgets.FloatSlider(
        min=0, max=6, step=0.25, value=2.5, description="posição",
        continuous_update=False,
    ),
    inclinacao_1=angulo_do_corte("inclinação 1"),
    inclinacao_2=angulo_do_corte("inclinação 2"),
    deslocamento=widgets.FloatSlider(
        min=-2, max=2, step=0.25, value=0, description="deslocamento",
        continuous_update=False,
    ),
);

interactive(children=(RadioButtons(description='modo', options=('Perpendicular à diagonal AG', 'Corte livre'),…

**Pare e pense**

1. Por que é impossível obter um heptágono cortando um cubo com um plano?
2. No modo 1, para quais posições a seção é um triângulo? E um hexágono?

<details>
<summary>Ver respostas</summary>

1. Cada lado da seção está numa face do cubo, e o plano corta cada face no máximo uma vez. O cubo tem 6 faces, então a seção tem no máximo 6 lados.
2. Triângulo para posições entre 0 e 2 (inclusive 2, que dá o triângulo $BDE$) e entre 4 e 6; hexágono para posições entre 2 e 4. Em 3, o hexágono é regular.
</details>

## Resumo

**Prisma:** duas bases congruentes em planos paralelos; faces laterais paralelogramos; arestas laterais paralelas e iguais. Base com $n$ lados: $V = 2n$, $A = 3n$, $F = n + 2$.

**Classificação:** reto (arestas laterais ⟂ bases; $h = \ell$), oblíquo ($h = \ell \sin\theta$), regular (reto com base regular).

**Paralelepípedo:** prisma de bases paralelogramos. Faces opostas paralelas e congruentes; as 4 diagonais se cortam no ponto médio. Retângulo (faces retângulos), cubo (arestas iguais), romboedro (faces losangos congruentes).

**Paralelepípedo retângulo** $a \times b \times c$: $d = \sqrt{a^2 + b^2 + c^2}$; $S = 2(ab + ac + bc)$; $(a + b + c)^2 = d^2 + S$.

**Cubo** de aresta $a$: diagonal da face $a\sqrt{2}$; diagonal $a\sqrt{3}$; $S = 6a^2$.

**Áreas do prisma:** $S_\ell = 2p \cdot h$ (reto); $S_\ell = 2p_{\text{seção reta}} \cdot \ell$ (qualquer prisma); $S_t = S_\ell + 2S_b$.

**Seções planas do cubo:** triângulo, quadrilátero, pentágono ou hexágono (nunca mais de 6 lados). Plano perpendicular à diagonal pelo centro: hexágono regular.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**.

O gabarito comentado está em [`0608b_prismas_e_paralelepipedos_gabarito.ipynb`](0608b_prismas_e_paralelepipedos_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Todo prisma regular é reto.
# (b) As faces laterais de um prisma oblíquo são retângulos.
# (c) Todo cubo é um paralelepípedo retângulo.
# (d) Todo romboedro é um cubo.
# (e) Um prisma hexagonal tem 18 arestas.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": False, "c": True, "d": False, "e": True}
secoes_ex1 = {
    "a": "seção 1 - Prismas",
    "b": "seção 1 - Prismas",
    "c": "seção 2 - Paralelepípedos e romboedros",
    "d": "seção 2 - Paralelepípedos e romboedros",
    "e": "seção 1 - Prismas",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; releia as "
        f"classificações das seções 1 e 2 ({secoes_ex1[item]})."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: o cubo. Um cubo tem aresta 3. Calcule, em Python (use
# math.sqrt):
# (a) a diagonal de uma face;
# (b) a diagonal do cubo;
# (c) a área total.
diagonal_face_ex2 = ...
diagonal_cubo_ex2 = ...
area_ex2 = ...

print(diagonal_face_ex2, diagonal_cubo_ex2, area_ex2)

In [ ]:
# Verificação
respostas_ex2 = [
    ("a", diagonal_face_ex2, 3 * math.sqrt(2)),
    ("b", diagonal_cubo_ex2, 3 * math.sqrt(3)),
    ("c", area_ex2, 54),
]
for item, resposta, certa in respostas_ex2:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), diagonal da face = a√2, diagonal "
        "do cubo = a√3, área = 6a² (seção 3 - Diagonal e área do paralelepípedo retângulo e do cubo)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: o paralelepípedo retângulo. As dimensões são 3, 4 e 12.
# Calcule:
# (a) a diagonal;
# (b) a área total.
diagonal_ex3 = ...
area_ex3 = ...

print(diagonal_ex3, area_ex3)

In [ ]:
# Verificação
respostas_ex3 = [("a", diagonal_ex3, 13), ("b", area_ex3, 192)]
for item, resposta, certa in respostas_ex3:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), d = √(a² + b² + c²) e "
        "S = 2(ab + ac + bc) (seção 3 - Diagonal e área do paralelepípedo retângulo e do cubo)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: a vassoura. Uma caixa tem 1,0 m × 0,8 m × 0,6 m.
# (a) Calcule a diagonal da caixa, em metros.
# (b) Uma vassoura de 1,5 m cabe na caixa? Responda True ou False.
# (c) Uma vassoura de 1,4 m cabe na caixa? Responda True ou False.
diagonal_ex4 = ...
cabe_15_ex4 = ...
cabe_14_ex4 = ...

print(diagonal_ex4, cabe_15_ex4, cabe_14_ex4)

In [ ]:
# Verificação
assert diagonal_ex4 is not ... and abs(diagonal_ex4 - math.sqrt(2)) <= 0.01, (
    "Tente de novo, revise: no item (a), use d = √(a² + b² + c²) com as medidas da "
    "caixa (seção 3 - Diagonal e área do paralelepípedo retângulo e do cubo)."
)
assert cabe_15_ex4 is False and cabe_14_ex4 is True, (
    "Tente de novo, revise: nos itens (b) e (c), compare o comprimento da vassoura "
    "com a diagonal da caixa (seção 3 - Diagonal e área do paralelepípedo retângulo e do cubo)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: o prisma hexagonal. Um prisma hexagonal regular tem
# aresta da base 2 cm e altura 5 cm. Calcule:
# (a) a área lateral;
# (b) a área de uma base;
# (c) a área total.
area_lateral_ex5 = ...
area_base_ex5 = ...
area_total_ex5 = ...

print(area_lateral_ex5, area_base_ex5, area_total_ex5)

In [ ]:
# Verificação
respostas_ex5 = [
    ("a", area_lateral_ex5, 60),
    ("b", area_base_ex5, 6 * math.sqrt(3)),
    ("c", area_total_ex5, 60 + 12 * math.sqrt(3)),
]
for item, resposta, certa in respostas_ex5:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), a base é formada por 6 triângulos "
        "equiláteros de lado 2 (seção 4 - Área lateral e área total do prisma)."
    )
print("Certo!")

In [ ]:
# Exercício Intermediário 6: área sem as dimensões. A soma das três dimensões de um
# paralelepípedo retângulo é 12 e a sua diagonal mede √50. Qual é a área total?
area_ex6 = ...

print(area_ex6)

In [ ]:
# Verificação
assert area_ex6 is not ... and abs(area_ex6 - 94) <= 0.01, (
    "Tente de novo, revise: use (a + b + c)² = d² + S (seção 3 - Diagonal e área do paralelepípedo retângulo e do cubo)."
)
print("Certo!")
print("Um paralelepípedo assim é o de dimensões 3, 4 e 5 (confira!).")

### Desafio

In [ ]:
# Exercício Desafio 7: o hexágono escondido. Um cubo de aresta 2 é cortado por um
# plano perpendicular a uma diagonal, passando pelo centro. Calcule:
# (a) o lado do hexágono regular obtido;
# (b) a área desse hexágono.
# Dica: o plano passa pelos pontos médios de seis arestas. Dois pontos médios
# vizinhos estão em arestas que saem de um mesmo vértice, a 1 unidade dele em cada
# direção.
lado_ex7 = ...
area_ex7 = ...

print(lado_ex7, area_ex7)

In [ ]:
# Verificação
respostas_ex7 = [("a", lado_ex7, math.sqrt(2)), ("b", area_ex7, 3 * math.sqrt(3))]
for item, resposta, certa in respostas_ex7:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), o lado é a hipotenusa de um "
        "triângulo retângulo de catetos 1 e 1; a área do hexágono é 6 vezes a de um "
        "triângulo equilátero (seção 5 - Seções planas do cubo)."
    )
print("Certo!")

# Conferência com o utils_3d: o plano pelo centro (1, 1, 1), perpendicular a AG.
cubo = u3d.paralelepipedo_retangulo(2, 2, 2)
hexagono = u3d.secao_plana(*cubo, ((1, 1, 1), (1, 1, 1)))
print(f"Lado medido: {u3d.medir_distancia(hexagono[0], hexagono[1]):.4f}")
print(f"Área medida: {u3d.medir_area_do_poligono(hexagono):.4f}")
fig = u3d.criar_figura(titulo="O hexágono regular escondido no cubo")
fig = u3d.desenhar_poliedro(fig, *cubo, opacidade=0.15, rotulos=list("ABCDEFGH"))
fig = u3d.desenhar_poligono(fig, hexagono, cor=u3d.CORES["destaque"], opacidade=0.7)
u3d.mostrar(fig)

In [9]:
# Figura de apoio do exercício 8: a caixa 3 × 4 × 12, com os cantos A e G.
caixa = u3d.vertices_do_paralelepipedo(3, 4, 12)
fig = u3d.criar_figura(titulo="A formiga sai de A e quer chegar a G")
fig = u3d.desenhar_poliedro(
    fig, *u3d.paralelepipedo_retangulo(3, 4, 12), opacidade=0.25, rotulos=list(caixa)
)
fig = u3d.desenhar_ponto(fig, caixa["A"], cor=u3d.CORES["destaque"], tamanho=8)
fig = u3d.desenhar_ponto(fig, caixa["G"], cor=u3d.CORES["destaque"], tamanho=8)
u3d.mostrar(fig)

In [ ]:
# Exercício Desafio 8: o caminho da formiga. Uma formiga está no vértice A de uma
# caixa (paralelepípedo retângulo) com AB = 3, AD = 4 e AE = 12, e quer chegar ao
# vértice G andando SOBRE A SUPERFÍCIE da caixa. Qual é o menor caminho possível?
# Dica: planifique duas faces vizinhas da caixa, formando um retângulo: o caminho
# mais curto sobre elas é a diagonal desse retângulo. Há três formas de escolher as
# duas faces; calcule as três e fique com a menor.
caminho_ex8 = ...

print(caminho_ex8)

In [ ]:
# Verificação
assert caminho_ex8 is not ... and abs(caminho_ex8 - math.sqrt(193)) <= 0.01, (
    "Tente de novo, revise: os três candidatos são √((3+4)² + 12²), √((3+12)² + 4²) "
    "e √((4+12)² + 3²) (seção 3 - Diagonal e área do paralelepípedo retângulo e do cubo)."
)
print("Certo!")

# O caminho da formiga: ele cruza a aresta BF num ponto P. Na planificação das faces
# ABFE e BCGF, o caminho é a diagonal de um retângulo 7 × 12; por semelhança de
# triângulos, P fica a 12 · 3/7 de B, subindo pela aresta BF.
caixa = u3d.vertices_do_paralelepipedo(3, 4, 12)
P = (3, 0, 12 * 3 / 7)
fig = u3d.criar_figura(titulo="Andando (por fora) × voando (por dentro)")
fig = u3d.desenhar_poliedro(
    fig, *u3d.paralelepipedo_retangulo(3, 4, 12), opacidade=0.2, rotulos=list(caixa)
)
fig = u3d.desenhar_segmento(fig, caixa["A"], P, cor=u3d.CORES["destaque"], espessura=6)
fig = u3d.desenhar_segmento(fig, P, caixa["G"], cor=u3d.CORES["destaque"], espessura=6)
fig = u3d.desenhar_ponto(fig, P, rotulo="P")
fig = u3d.desenhar_segmento(
    fig, caixa["A"], caixa["G"], cor=u3d.CORES["apoio"], tracejado=True
)
u3d.mostrar(fig)
andando = u3d.medir_distancia(caixa["A"], P) + u3d.medir_distancia(P, caixa["G"])
print(f"Por fora (andando): {andando:.2f}")
print(f"Por dentro (voando): {u3d.medir_distancia(caixa['A'], caixa['G']):.2f}")
print("Por dentro (voando), o caminho seria 13; por fora (andando), o menor é cerca de 13,89.")

## Conexões

**Isso depende de:**
- [`0602a_paralelismo.ipynb`](0602a_paralelismo.ipynb) (arestas laterais paralelas, bases em planos paralelos)
- [`0603a_perpendicularidade.ipynb`](0603a_perpendicularidade.ipynb) (aresta perpendicular ao plano da base, na diagonal do paralelepípedo; a diagonal do cubo perpendicular ao plano $BDE$)
- [`0604a_projecoes_distancias_e_angulos.ipynb`](0604a_projecoes_distancias_e_angulos.ipynb) (altura como distância entre planos; ângulo entre a aresta lateral e a base)
- [`0605a_diedros.ipynb`](0605a_diedros.ipynb) (a seção reta, como no diedro)
- [`0607a_poliedros_convexos.ipynb`](0607a_poliedros_convexos.ipynb) (Relação de Euler e planificações)
- [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (Teorema de Pitágoras)
- [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (polígonos regulares e apótema)
- [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (áreas de polígonos)

**Isso será usado em:**
- [`0609a_volume_e_principio_de_cavalieri.ipynb`](0609a_volume_e_principio_de_cavalieri.ipynb) (volume do paralelepípedo, do cubo e do prisma)
- [`0610a_piramide.ipynb`](0610a_piramide.ipynb) (a pirâmide comparada ao prisma de mesma base e altura)
- [`0611a_cilindro.ipynb`](0611a_cilindro.ipynb) (o cilindro como "prisma de infinitos lados")
- 📌 Sólidos semelhantes e troncos (tronco de prisma triangular)
- 📌 Inscrição e circunscrição: esfera e poliedros (a diagonal do cubo e a esfera circunscrita)